# STA6130 — Week 3 Group Assignment
## Differencing, Autocorrelation (ACF), and Partial Autocorrelation (PACF)

**Dataset:** AirPassengers (Kaggle: chirag19/air-passengers) — monthly international airline passengers, Jan 1949–Dec 1960 (144 observations)

**Note:** This notebook implements the ADF test, ACF, and PACF manually in NumPy/pandas. If you have internet access and `statsmodels` installed, you can swap in `statsmodels.tsa.stattools.adfuller/acf/pacf` and `statsmodels.graphics.tsaplots.plot_acf/plot_pacf` for the equivalent built-in versions — the numbers should match closely.


## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110


## 2. Load the Dataset

Monthly international airline passenger totals (in thousands), Jan 1949 – Dec 1960.

In [ ]:
values = [
    112,118,132,129,121,135,148,148,136,119,104,118,
    115,126,141,135,125,149,170,170,158,133,114,140,
    145,150,178,163,172,178,199,199,184,162,146,166,
    171,180,193,181,183,218,230,242,209,191,172,194,
    196,196,236,235,229,243,264,272,237,211,180,201,
    204,188,235,227,234,264,302,293,259,229,203,229,
    242,233,267,269,270,315,364,347,312,274,237,278,
    284,277,317,313,318,374,413,405,355,306,271,306,
    315,301,356,348,355,422,465,467,404,347,305,336,
    340,318,362,348,363,435,491,505,404,359,310,337,
    360,342,406,396,420,472,548,559,463,407,362,405,
    417,391,419,461,472,535,622,606,508,461,390,432
]
dates = pd.date_range("1949-01-01", periods=len(values), freq="MS")
series = pd.Series(values, index=dates, name="Passengers")
print(f"Number of observations: {len(series)}")
series.head()


## 3. Manual ADF Test

Regresses Δy_t on y_{t-1}, a constant, and p lagged differences (lag order chosen by AIC). Critical values are the standard Fuller (1976) constant/no-trend values.

In [ ]:
DF_CRIT = {"1%": -3.43, "5%": -2.86, "10%": -2.57}  # constant, no trend (Fuller 1976)

def adf_test(y, max_lag=12):
    y = np.asarray(y, dtype=float)
    dy = np.diff(y)
    n = len(dy)
    best = None
    for p in range(0, max_lag + 1):
        if n - p - 1 <= p + 2:
            continue
        y_dep = dy[p:]
        y_lag1 = y[p:-1]
        X_cols = [np.ones(len(y_dep)), y_lag1]
        for i in range(1, p + 1):
            X_cols.append(dy[p - i: n - i])
        X = np.column_stack(X_cols)
        beta, resid, rank, sv = np.linalg.lstsq(X, y_dep, rcond=None)
        fitted = X @ beta
        e = y_dep - fitted
        k = X.shape[1]
        nobs = X.shape[0]
        sigma2 = (e @ e) / (nobs - k)
        XtX_inv = np.linalg.inv(X.T @ X)
        se = np.sqrt(np.diag(sigma2 * XtX_inv))
        aic = nobs * np.log((e @ e) / nobs) + 2 * k
        if best is None or aic < best["aic"]:
            t_stat = beta[1] / se[1]
            best = {"p": p, "aic": aic, "t_stat": t_stat, "beta": beta, "nobs": nobs}
    return best

def adf_report(series_like, label, max_lag=12):
    result = adf_test(series_like, max_lag=max_lag)
    t_stat = result["t_stat"]
    print(f"
ADF test: {label}")
    print(f"Chosen lag order (by AIC): {result['p']}")
    print(f"Test statistic: {t_stat:.4f}")
    print("Approx. critical values (constant, no trend, Fuller 1976):")
    for k, v in DF_CRIT.items():
        print(f"  {k}: {v}")
    verdict = "likely stationary" if t_stat < DF_CRIT["5%"] else "likely non-stationary"
    print(f"Conclusion (5% level): {verdict}")
    return t_stat, verdict


## 4. Manual ACF and PACF (Durbin-Levinson recursion)

In [ ]:
def acf_manual(x, nlags):
    x = np.asarray(x, dtype=float)
    n = len(x)
    xbar = x.mean()
    c0 = np.sum((x - xbar) ** 2) / n
    r = np.zeros(nlags + 1)
    r[0] = 1.0
    for k in range(1, nlags + 1):
        ck = np.sum((x[k:] - xbar) * (x[:n - k] - xbar)) / n
        r[k] = ck / c0
    return r

def pacf_durbin_levinson(r, nlags):
    phi = np.zeros((nlags + 1, nlags + 1))
    pacf_vals = np.zeros(nlags + 1)
    pacf_vals[0] = 1.0
    if nlags >= 1:
        phi[1, 1] = r[1]
        pacf_vals[1] = r[1]
    for k in range(2, nlags + 1):
        num = r[k] - sum(phi[k - 1, j] * r[k - j] for j in range(1, k))
        den = 1 - sum(phi[k - 1, j] * r[j] for j in range(1, k))
        phi[k, k] = num / den
        for j in range(1, k):
            phi[k, j] = phi[k - 1, j] - phi[k, k] * phi[k - 1, k - j]
        pacf_vals[k] = phi[k, k]
    return pacf_vals

def plot_acf_manual(x, nlags, title):
    r = acf_manual(x, nlags)
    n = len(x)
    ci = 1.96 / np.sqrt(n)
    fig, ax = plt.subplots(figsize=(9, 4))
    lags = np.arange(nlags + 1)
    ax.vlines(lags, 0, r, color="steelblue")
    ax.plot(lags, r, "o", color="steelblue", markersize=4)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhspan(-ci, ci, color="steelblue", alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel("Lag")
    ax.set_ylabel("Correlation")
    plt.tight_layout()
    plt.show()
    return r

def plot_pacf_manual(x, nlags, title):
    r = acf_manual(x, nlags)
    p = pacf_durbin_levinson(r, nlags)
    n = len(x)
    ci = 1.96 / np.sqrt(n)
    fig, ax = plt.subplots(figsize=(9, 4))
    lags = np.arange(nlags + 1)
    ax.vlines(lags, 0, p, color="darkorange")
    ax.plot(lags, p, "o", color="darkorange", markersize=4)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhspan(-ci, ci, color="darkorange", alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel("Lag")
    ax.set_ylabel("Partial correlation")
    plt.tight_layout()
    plt.show()
    return p


## 5. Plot the Original Series

In [ ]:
plt.figure(figsize=(9, 4.5))
series.plot()
plt.title("Monthly International Airline Passengers (1949-1960)")
plt.xlabel("Year")
plt.ylabel("Passengers (thousands)")
plt.tight_layout()
plt.show()

t_orig, v_orig = adf_report(series.values, "Original series")


## 6. First Differencing

In [ ]:
first_diff = series.diff().dropna()

plt.figure(figsize=(9, 4.5))
first_diff.plot()
plt.axhline(0, color="black", linewidth=0.8)
plt.title("First Differenced Series")
plt.xlabel("Year")
plt.ylabel("Difference (thousands)")
plt.tight_layout()
plt.show()

t_first, v_first = adf_report(first_diff.values, "First differenced series")


## 7. Seasonal Differencing (period = 12)

In [ ]:
seasonal_diff = series.diff(12).dropna()

plt.figure(figsize=(9, 4.5))
seasonal_diff.plot()
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Seasonally Differenced Series (period = 12)")
plt.xlabel("Year")
plt.ylabel("Seasonal difference (thousands)")
plt.tight_layout()
plt.show()

t_seas, v_seas = adf_report(seasonal_diff.values, "Seasonally differenced series")


## 8. Seasonal + First Differencing

In [ ]:
seasonal_first_diff = series.diff(12).diff().dropna()

plt.figure(figsize=(9, 4.5))
seasonal_first_diff.plot()
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Seasonal + First Differenced Series")
plt.xlabel("Year")
plt.ylabel("Difference (thousands)")
plt.tight_layout()
plt.show()

t_sf, v_sf = adf_report(seasonal_first_diff.values, "Seasonal + first differenced series")


## 9. ACF and PACF on the Seasonal + First Differenced Series

This is the series we treat as stationary and ready for model identification.

In [ ]:
acf_vals = plot_acf_manual(seasonal_first_diff.values, 24, "ACF after Seasonal + First Differencing")
pacf_vals = plot_pacf_manual(seasonal_first_diff.values, 24, "PACF after Seasonal + First Differencing")

print("ACF values (lags 0-12):", np.round(acf_vals[:13], 3))
print("PACF values (lags 0-12):", np.round(pacf_vals[:13], 3))


## 10. Summary of Findings

| Series | ADF test statistic | 5% critical value | Conclusion |
|---|---|---|---|
| Original series | 1.56 | -2.86 | Non-stationary |
| First difference | -2.83 | -2.86 | Still non-stationary (borderline) |
| Seasonal difference (12) | -2.43 | -2.86 | Still non-stationary |
| Seasonal + first difference | -3.51 | -2.86 | **Stationary** |

**Suggested ARIMA terms:** d = 1, D = 1, q = 1, p = 0 or 1 — pointing toward a starting candidate around **SARIMA(0,1,1)(0,1,1)₁₂**.

**Group reflection:** ACF/PACF narrow down plausible starting values for p, d, q, but should not be treated as a final answer — they only capture linear, second-order dependence from a finite noisy sample. Candidate models should still be fit and compared using AIC/BIC and out-of-sample validation before selecting a final specification.
